# Lesson 12 — RAG Agent (Retrieval-Augmented Generation)

## What you will learn
- What **RAG** is and why it prevents hallucinations
- How to embed documents into a **local vector store** (Chroma)
- How to build a `retrieve_docs` **tool** the agent calls automatically
- **Agentic RAG**: the agent decides *when* to retrieve
- **Self-correcting RAG**: check relevance, re-query if context is insufficient

## Mental model
```
User question
   ↓
[agent] → retrieve_docs(query) → [vector store]
   ↑              ↓ retrieved chunks
   └──── reads context ──────────┘
   ↓
[grounded answer based on real documents]
```

> **Without RAG:** LLM answers from training data → may hallucinate  
> **With RAG:** LLM answers from *your* retrieved documents → grounded

In [1]:
# Install dependencies if needed
# !pip install langgraph langchain-ollama langchain-community chromadb

## Step 1 — Build the Knowledge Base

In production this would be your PDFs, docs, or database records.  
Here we use a hardcoded list of facts about LangGraph.

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))
from config import get_ollama_model

import os
from typing import Annotated
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

try:
    from langchain_chroma import Chroma
    CHROMA_AVAILABLE = True
    print("✅ Chroma available")
except ImportError:
    CHROMA_AVAILABLE = False
    print("⚠️  Install: pip install langchain-community chromadb")

KNOWLEDGE_BASE = [
    "LangGraph is a library for building stateful, multi-step AI agent workflows as directed graphs.",
    "A StateGraph defines the workflow. Nodes are Python functions. Edges define execution order.",
    "The state is a TypedDict shared between all nodes. Reducers control how fields are updated.",
    "add_messages is a reducer that appends messages to a list instead of replacing them.",
    "compile() validates the graph and returns a runnable. invoke() runs the graph synchronously.",
    "stream() runs the graph and yields state snapshots after each node — used for real-time UIs.",
    "The @tool decorator creates tools. The docstring tells the LLM when and how to use the tool.",
    "ToolNode is a built-in node that automatically executes all tool calls from the last message.",
    "Human-in-the-loop uses interrupt() to pause execution and wait for human input.",
    "SqliteSaver and PostgresSaver persist graph state to disk across Python restarts.",
    "Subgraphs are compiled graphs used as nodes inside parent graphs.",
    "Send() launches multiple node executions in parallel, each with its own state copy.",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")

✅ Chroma available
Knowledge base: 12 documents


## Step 2 — Embed and Index Documents

`OllamaEmbeddings` converts text → vectors. `Chroma` stores and indexes them for similarity search.

In [3]:
if CHROMA_AVAILABLE:
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    docs = [Document(page_content=text) for text in KNOWLEDGE_BASE]

    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
    chunks = splitter.split_documents(docs)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_db_l12",
        collection_name="langgraph_docs",
    )
    print(f"✅ Indexed {len(chunks)} chunks into Chroma")
else:
    vector_store = None
    print("Skipping — Chroma not available")

✅ Indexed 12 chunks into Chroma


## Step 3 — Build the Retrieval Tool

The agent calls `retrieve_docs(query)` when it needs factual grounding.
The tool returns the top-3 most relevant chunks.

In [4]:
@tool
def retrieve_docs(query: str) -> str:
    """Search the knowledge base for information relevant to the query.
    Use this for any question about LangGraph concepts or features."""
    if not CHROMA_AVAILABLE or vector_store is None:
        return "Knowledge base not available. Install chromadb first."
    results = vector_store.similarity_search(query, k=3)
    if not results:
        return "No relevant documents found."
    formatted = "\n\n".join([f"[Doc {i+1}]: {r.page_content}" for i, r in enumerate(results)])
    print(f"  [retrieve_docs] Found {len(results)} chunks for: '{query[:50]}'")
    return formatted

# Test the tool directly
if CHROMA_AVAILABLE:
    result = retrieve_docs.invoke({"query": "what is state in langgraph?"})
    print(result[:300])

  [retrieve_docs] Found 3 chunks for: 'what is state in langgraph?'
[Doc 1]: The state is a TypedDict shared between all nodes. Reducers control how fields are updated.

[Doc 2]: LangGraph is a library for building stateful, multi-step AI agent workflows as directed graphs.

[Doc 3]: stream() runs the graph and yields state snapshots after each node — used for real-


## Step 4 — Build the RAG Agent Graph

In [5]:
tools = [retrieve_docs]
llm = ChatOllama(model=get_ollama_model(), temperature=0)
llm_with_tools = llm.bind_tools(tools)

class RAGState(TypedDict):
    messages: Annotated[list, add_messages]

SYSTEM = SystemMessage(content=(
    "You are a LangGraph expert assistant. "
    "ALWAYS call retrieve_docs before answering any question about LangGraph. "
    "Base your answer ONLY on the retrieved documents."
))

def agent_node(state: RAGState) -> dict:
    print(f"[agent] thinking... ({len(state['messages'])} messages)")
    response = llm_with_tools.invoke([SYSTEM] + state["messages"])
    return {"messages": [response]}

def should_continue(state: RAGState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"

builder = StateGraph(RAGState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")
graph = builder.compile()

print("RAG agent compiled!")

RAG agent compiled!


## Step 5 — Run the RAG Agent

In [6]:
def ask(question: str):
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    result = graph.invoke({"messages": [HumanMessage(content=question)]})
    print(f"\nA: {result['messages'][-1].content}")

ask("What is the difference between invoke() and stream()?")  
ask("How does the add_messages reducer work?")
ask("What is Send() used for?")


Q: What is the difference between invoke() and stream()?
[agent] thinking... (1 messages)
  [retrieve_docs] Found 3 chunks for: 'difference between invoke() and stream()'
[agent] thinking... (3 messages)

A: The difference between `invoke()` and `stream()` in LangGraph is as follows:

- **`invoke()`**:  
  - Runs the graph **synchronously**. It executes the graph in a single pass, returning a **runnable** object that can be used to execute further operations.  
  - Example: Used for real-time applications or when a user needs to interact with the graph in sequence.

- **`stream()`**:  
  - Runs the graph **asynchronously**. It yields **state snapshots** after each node, ideal for real-time UIs or applications that require live data updates.  
  - Example: Used for applications that need to display data dynamically, such as live dashboards or interactive interfaces.

Both functions are part of LangGraph's capabilities to manage and execute graph-based workflows, with distinct behaviors

## Key Takeaways

| Concept | Detail |
|---------|--------|
| `OllamaEmbeddings` | Converts text → numeric vectors locally |
| `Chroma.from_documents()` | Builds and persists the vector index |
| `similarity_search(query, k=3)` | Returns top-k most semantically similar chunks |
| `@tool` on retrieve function | Lets agent decide when to call it |
| System prompt ALWAYS retrieve | Forces grounding on every answer |

## 🏋️ Exercise
1. Add 5 more documents to `KNOWLEDGE_BASE` about Python best practices
2. Add a second tool `retrieve_python_docs(query)` that searches a **separate** Chroma collection
3. Ask a mixed question: "How does LangGraph work and what Python features does it rely on?"
4. Observe the agent calling both tools

In [ ]:
# Your exercise solution here
